**Setup**

In [3]:
from google.colab import userdata
import os
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
path = os.path.expanduser('~/.kaggle/access_token')
with open(path, 'w') as f:
    f.write(userdata.get('KAGGLE_API_TOKEN'))
os.chmod(path, 0o600)

!kaggle datasets download -d rohanrao/sudoku
!unzip -q sudoku.zip
print("Dataset ready")

Dataset URL: https://www.kaggle.com/datasets/rohanrao/sudoku
License(s): CC0-1.0
sudoku.zip: Skipping, found more recently modified local copy (use --force to force download)
replace sudoku.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
Dataset ready


**Load data and helpers**

In [4]:
import time, random, heapq, itertools
import pandas as pd

N = 20   # number of puzzles to test; never load the whole file
df = pd.read_csv('sudoku.csv', nrows=N)

def string_to_board(s):
    return [[int(ch) for ch in s[i:i+9]] for i in range(0, 81, 9)]

def board_to_string(b):
    return ''.join(str(v) for row in b for v in row)

def print_board(board):
    for row in board:
        print(" ".join(str(c) if c != 0 else '.' for c in row))

def is_valid(board, row, col, num):
    for i in range(9):
        if board[row][i] == num or board[i][col] == num:
            return False
    sr, sc = 3*(row//3), 3*(col//3)
    for i in range(3):
        for j in range(3):
            if board[sr+i][sc+j] == num:
                return False
    return True

def candidates(board, r, c):
    used = set(board[r]) | {board[i][c] for i in range(9)}
    sr, sc = 3*(r//3), 3*(c//3)
    used |= {board[sr+i][sc+j] for i in range(3) for j in range(3)}
    return [n for n in range(1, 10) if n not in used]

class SearchLimit(Exception):
    pass

sample = df.iloc[0]['puzzle']
print("Initial Sudoku Puzzle:")
print_board(string_to_board(sample))

Initial Sudoku Puzzle:
. 7 . . . . . 4 3
. 4 . . . 9 6 1 .
8 . . 6 3 4 9 . .
. 9 4 . 5 2 . . .
3 5 8 4 6 . . 2 .
. . . 8 . . 5 3 .
. 8 . . 7 . . 9 1
9 . 2 1 . . . . 5
. . 7 . 4 . 8 . 2


**Backtracking**

In [5]:
def solve_backtracking(initial_board, max_nodes=2_000_000):
    board = [row[:] for row in initial_board]
    stats = {'nodes': 0}

    def rec():
        for r in range(9):
            for c in range(9):
                if board[r][c] == 0:
                    for num in range(1, 10):
                        if is_valid(board, r, c, num):
                            stats['nodes'] += 1
                            if stats['nodes'] > max_nodes:
                                raise SearchLimit
                            board[r][c] = num
                            if rec():
                                return True
                            board[r][c] = 0
                    return False
        return True

    try:
        ok = rec()
    except SearchLimit:
        ok = False
    return (board if ok else None), stats

res, st = solve_backtracking(string_to_board(sample))
print("\nBacktracking:", st); print_board(res)


Backtracking: {'nodes': 145}
6 7 9 5 1 8 2 4 3
5 4 3 7 2 9 6 1 8
8 2 1 6 3 4 9 5 7
7 9 4 3 5 2 1 8 6
3 5 8 4 6 1 7 2 9
2 1 6 8 9 7 5 3 4
4 8 5 2 7 6 3 9 1
9 6 2 1 8 3 4 7 5
1 3 7 9 4 5 8 6 2


**[A] Search**

In [6]:
def heuristic(board):
    # h = number of empty cells (admissible: each move fills exactly one cell, cost 1)
    return sum(row.count(0) for row in board)

def solve_a_star(initial_board, max_nodes=200_000):
    tie = itertools.count()                     # avoids comparing boards on ties
    start = [row[:] for row in initial_board]
    # queue entries: (f, -g, tie, board). -g => deeper states first on equal f
    pq = [(heuristic(start), 0, next(tie), start)]
    stats = {'nodes': 0, 'max_queue': 1}

    while pq:
        f, neg_g, _, cur = heapq.heappop(pq)
        g = -neg_g
        stats['nodes'] += 1
        if stats['nodes'] > max_nodes:
            return None, stats

        # pick the most constrained empty cell (fewest candidates)
        best, best_c, dead = None, None, False
        for r in range(9):
            for c in range(9):
                if cur[r][c] == 0:
                    cands = candidates(cur, r, c)
                    if not cands:
                        dead = True
                        break
                    if best_c is None or len(cands) < len(best_c):
                        best, best_c = (r, c), cands
            if dead:
                break
        if dead:
            continue                            # prune: this state cannot be solved
        if best is None:
            return cur, stats                   # goal: no empty cells

        r, c = best
        for num in best_c:
            nb = [row[:] for row in cur]
            nb[r][c] = num
            heapq.heappush(pq, (g + 1 + heuristic(nb), -(g + 1), next(tie), nb))
        stats['max_queue'] = max(stats['max_queue'], len(pq))
    return None, stats

res, st = solve_a_star(string_to_board(sample))
print("\nA*:", st); print_board(res)


A*: {'nodes': 45, 'max_queue': 1}
6 7 9 5 1 8 2 4 3
5 4 3 7 2 9 6 1 8
8 2 1 6 3 4 9 5 7
7 9 4 3 5 2 1 8 6
3 5 8 4 6 1 7 2 9
2 1 6 8 9 7 5 3 4
4 8 5 2 7 6 3 9 1
9 6 2 1 8 3 4 7 5
1 3 7 9 4 5 8 6 2


**Hill Climbing**

In [7]:
def _init_box_fill(initial):
    board = [row[:] for row in initial]
    free = []
    for br in range(0, 9, 3):
        for bc in range(0, 9, 3):
            cells = [(br+i, bc+j) for i in range(3) for j in range(3)]
            present = {board[r][c] for r, c in cells if board[r][c] != 0}
            missing = [n for n in range(1, 10) if n not in present]
            random.shuffle(missing)
            empties = [(r, c) for r, c in cells if board[r][c] == 0]
            for (r, c), v in zip(empties, missing):
                board[r][c] = v
            free.append(empties)
    return board, free

def _row_cost(b, r): return 9 - len(set(b[r]))
def _col_cost(b, c): return 9 - len({b[i][c] for i in range(9)})
def _local(b, rows, cols):
    return sum(_row_cost(b, r) for r in rows) + sum(_col_cost(b, c) for c in cols)
def total_conflicts(b):
    return sum(_row_cost(b, i) + _col_cost(b, i) for i in range(9))

def solve_hill_climbing(initial_board, max_restarts=200, max_steps=300,
                        max_sideways=30, time_limit=10, seed=None):
    if seed is not None:
        random.seed(seed)
    t0 = time.perf_counter()
    stats = {'nodes': 0, 'restarts': 0, 'final_conflicts': None}
    best_overall = None

    for _ in range(max_restarts):
        stats['restarts'] += 1
        board, free = _init_box_fill(initial_board)   # boxes always valid
        cost = total_conflicts(board)
        sideways = 0

        for _ in range(max_steps):
            if cost == 0:
                stats['final_conflicts'] = 0
                return board, stats
            if time.perf_counter() - t0 > time_limit:
                return None, stats

            best_delta, best_moves = None, []
            for cells in free:                         # neighbors: swap 2 free cells in a box
                for a in range(len(cells)):
                    for b in range(a+1, len(cells)):
                        (r1, c1), (r2, c2) = cells[a], cells[b]
                        if board[r1][c1] == board[r2][c2]:
                            continue
                        rows, cols = {r1, r2}, {c1, c2}
                        before = _local(board, rows, cols)
                        board[r1][c1], board[r2][c2] = board[r2][c2], board[r1][c1]
                        after = _local(board, rows, cols)
                        board[r1][c1], board[r2][c2] = board[r2][c2], board[r1][c1]
                        stats['nodes'] += 1
                        d = after - before
                        if best_delta is None or d < best_delta:
                            best_delta, best_moves = d, [(r1, c1, r2, c2)]
                        elif d == best_delta:
                            best_moves.append((r1, c1, r2, c2))

            if best_delta is None or best_delta > 0:
                break                                  # local minimum -> restart
            if best_delta == 0:
                sideways += 1
                if sideways > max_sideways:
                    break
            else:
                sideways = 0
            r1, c1, r2, c2 = random.choice(best_moves)
            board[r1][c1], board[r2][c2] = board[r2][c2], board[r1][c1]
            cost += best_delta

        best_overall = cost if best_overall is None else min(best_overall, cost)
        stats['final_conflicts'] = best_overall
    return None, stats

res, st = solve_hill_climbing(string_to_board(sample))
print("\nHill Climbing:", st)
print_board(res) if res else print("Not solved within limits")


Hill Climbing: {'nodes': 3115, 'restarts': 1, 'final_conflicts': 0}
6 7 9 5 1 8 2 4 3
5 4 3 7 2 9 6 1 8
8 2 1 6 3 4 9 5 7
7 9 4 3 5 2 1 8 6
3 5 8 4 6 1 7 2 9
2 1 6 8 9 7 5 3 4
4 8 5 2 7 6 3 9 1
9 6 2 1 8 3 4 7 5
1 3 7 9 4 5 8 6 2


**Benchmark and comparison**

In [8]:
algorithms = {
    'Backtracking': solve_backtracking,
    'A*': solve_a_star,
    'Hill Climbing': solve_hill_climbing,
}

records = []
for i, row in df.iterrows():
    for name, fn in algorithms.items():
        t = time.perf_counter()
        result, stats = fn(string_to_board(row['puzzle']))
        dt = time.perf_counter() - t
        solved = result is not None
        records.append({
            'puzzle': i, 'algorithm': name, 'solved': solved,
            'correct': solved and board_to_string(result) == row['solution'],
            'time_s': round(dt, 4), 'nodes': stats['nodes'],
        })

results = pd.DataFrame(records)
print(results.to_string(index=False))

summary = results.groupby('algorithm').agg(
    success_rate=('correct', 'mean'),
    avg_time_s=('time_s', 'mean'),
    avg_nodes=('nodes', 'mean'),
).round(4)
print("\nSummary:\n", summary)

 puzzle     algorithm  solved  correct  time_s  nodes
      0  Backtracking    True     True  0.0011    145
      0            A*    True     True  0.0039     45
      0 Hill Climbing    True     True  0.2251  32218
      1  Backtracking    True     True  0.0017    214
      1            A*    True     True  0.0037     43
      1 Hill Climbing    True     True  0.0161   2349
      2  Backtracking    True     True  0.0004     57
      2            A*    True     True  0.0034     41
      2 Hill Climbing    True     True  0.0131   1850
      3  Backtracking    True     True  0.0016    126
      3            A*    True     True  0.0035     43
      3 Hill Climbing    True     True  0.0941  10788
      4  Backtracking    True     True  0.0017    153
      4            A*    True     True  0.0035     40
      4 Hill Climbing    True     True  0.0137   1672
      5  Backtracking    True     True  0.0005     67
      5            A*    True     True  0.0038     45
      5 Hill Climbing    Tru